# Data Ingestion and Scenario Generation
This notebook pulls the raw German electricity dataset, computes daily aggregates, and creates three renewable energy scenarios: worst, average, and best case.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests

project_root = Path.cwd()
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

raw_csv = raw_dir / "germany_energy_weather_2012_2020.csv"

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_csv}")
print(f"Processed data path: {processed_dir}")

In [ ]:
if raw_csv.exists():
    print(f"Loading existing raw dataset from {raw_csv}")
    df = pd.read_csv(raw_csv)
    df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'])
else:
    print("Raw dataset not found. Downloading from OPSD...")
    opsd_url = "https://data.open-power-system-data.org/time_series/latest/time_series_60min_singleindex.csv"
    df = pd.read_csv(
        opsd_url,
        usecols=[
            "utc_timestamp",
            "DE_load_actual_entsoe_transparency",
            "DE_wind_generation_actual",
            "DE_solar_generation_actual",
        ],
        parse_dates=["utc_timestamp"],
    )
    df.rename(columns={
        "DE_load_actual_entsoe_transparency": "load_mw",
        "DE_wind_generation_actual": "wind_generation_mw",
        "DE_solar_generation_actual": "solar_generation_mw",
    }, inplace=True)
    df.to_csv(raw_csv, index=False)
    print(f"Saved raw dataset to {raw_csv}")

print(df.head())
print(df.shape)

In [ ]:
df['date'] = df['utc_timestamp'].dt.date

daily = df.groupby('date', as_index=False).agg({
    'load_mw': 'mean',
    'wind_generation_mw': 'mean',
    'solar_generation_mw': 'mean',
})

daily['date'] = pd.to_datetime(daily['date'])
daily['renewable_total_mw'] = daily['wind_generation_mw'] + daily['solar_generation_mw']
daily['shortfall_mw'] = daily['load_mw'] - daily['renewable_total_mw']
daily['surplus_mw'] = (daily['renewable_total_mw'] - daily['load_mw']).clip(lower=0)

daily.to_csv(processed_dir / 'daily_data.csv', index=False)
print(f"Saved daily aggregates to {processed_dir / 'daily_data.csv'}")

scenarios = {
    'worst': {'wind_factor': 0.70, 'solar_factor': 0.65},
    'average': {'wind_factor': 1.00, 'solar_factor': 1.00},
    'best': {'wind_factor': 1.30, 'solar_factor': 1.35},
}

scenario_frames = []
for name, params in scenarios.items():
    scenario = daily.copy()
    scenario['wind_generation_mw'] = scenario['wind_generation_mw'] * params['wind_factor']
    scenario['solar_generation_mw'] = scenario['solar_generation_mw'] * params['solar_factor']
    scenario['renewable_total_mw'] = scenario['wind_generation_mw'] + scenario['solar_generation_mw']
    scenario['shortfall_mw'] = scenario['load_mw'] - scenario['renewable_total_mw']
    scenario['surplus_mw'] = (scenario['renewable_total_mw'] - scenario['load_mw']).clip(lower=0)
    scenario['coverage_pct'] = (scenario['renewable_total_mw'] / scenario['load_mw'] * 100).clip(lower=0)
    scenario['scenario'] = name
    scenario_frames.append(scenario)

all_scenarios = pd.concat(scenario_frames, ignore_index=True)
all_scenarios.to_csv(processed_dir / 'scenario_forecasts.csv', index=False)
print(f"Saved scenario forecasts to {processed_dir / 'scenario_forecasts.csv'}")
print(all_scenarios.head())